**Imports**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)
%matplotlib inline

os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv('../data/raw/application_train.csv')
print(f"Loaded: {df.shape}")

Loaded: (307511, 122)


**Drop High-Missingness Building Features**

In [2]:
# All _AVG, _MODE, _MEDI building/apartment features
building_cols = [col for col in df.columns if col.endswith(('_AVG', '_MODE', '_MEDI'))]
print(f"Dropping {len(building_cols)} building feature columns")

df.drop(columns=building_cols, inplace=True)
print(f"Shape after drop: {df.shape}")

Dropping 47 building feature columns
Shape after drop: (307511, 75)


**Fix DAYS_BIRTH and DAYS_EMPLOYED**

In [ ]:
# DAYS_BIRTH: negative days relative to today, convert to age in years
df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).astype(int)

# DAYS_EMPLOYED: negative = days employed, 365243 = anomaly (unemployed/pensioner)
df['DAYS_EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['EMPLOYMENT_YEARS'] = (-df['DAYS_EMPLOYED'] / 365)

# Verify
print("AGE_YEARS range:", df['AGE_YEARS'].min(), "—", df['AGE_YEARS'].max())
print("EMPLOYMENT_YEARS anomalies flagged:", df['DAYS_EMPLOYED_ANOMALY'].sum())
print("EMPLOYMENT_YEARS nulls:", df['EMPLOYMENT_YEARS'].isnull().sum())

AGE_YEARS range: 20 — 69
EMPLOYMENT_YEARS anomalies flagged: 55374
EMPLOYMENT_YEARS nulls: 55374


**Engineer Key Financial Ratios**

In [4]:
# Credit-to-income ratio — core underwriting metric
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']

# Annuity-to-income ratio — monthly burden as % of income
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

# Credit-to-goods ratio — LTV proxy
df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']

# Loan term in months (implied)
df['LOAN_TERM_MONTHS'] = df['AMT_CREDIT'] / df['AMT_ANNUITY']

# Income per family member
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS'].replace(0, 1)

print("Engineered financial ratios:")
print(df[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 
          'CREDIT_GOODS_RATIO', 'LOAN_TERM_MONTHS', 
          'INCOME_PER_PERSON']].describe())

Engineered financial ratios:
       CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO  CREDIT_GOODS_RATIO  \
count           307511.000            307499.000          307233.000   
mean                 3.958                 0.181               1.123   
std                  2.690                 0.095               0.124   
min                  0.005                 0.000               0.150   
25%                  2.019                 0.115               1.000   
50%                  3.265                 0.163               1.119   
75%                  5.160                 0.229               1.198   
max                 84.737                 1.876               6.000   

       LOAN_TERM_MONTHS  INCOME_PER_PERSON  
count        307499.000         307509.000  
mean             21.612          93105.880  
std               7.824         101373.363  
min               8.037           2812.500  
25%              15.614          47250.000  
50%              20.000          75000.000  
75%    

**Car and Document Flags**

In [ ]:
# HAS_CAR: missing OWN_CAR_AGE means no car
df['HAS_CAR'] = df['OWN_CAR_AGE'].notna().astype(int)
df.drop(columns=['OWN_CAR_AGE'], inplace=True)

# Document submission count, how many docs did the applicant provide?
doc_cols = [col for col in df.columns if col.startswith('FLAG_DOCUMENT')]
df['DOCUMENT_COUNT'] = df[doc_cols].sum(axis=1)
print(f"Document columns summed: {len(doc_cols)}")
print(f"HAS_CAR distribution:\n{df['HAS_CAR'].value_counts()}")

Document columns summed: 20
HAS_CAR distribution:
HAS_CAR
0    202929
1    104582
Name: count, dtype: int64


**Encode Categoricals**

In [6]:
from sklearn.preprocessing import LabelEncoder

cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"Encoding {len(cat_cols)} categorical columns: {cat_cols}")

le = LabelEncoder()
for col in cat_cols:
    df[col] = df[col].fillna('MISSING')
    df[col] = le.fit_transform(df[col])

print("All categoricals encoded.")

Encoding 12 categorical columns: ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE']
All categoricals encoded.


**Impute Remaining Numeric Nulls**

In [7]:
from sklearn.impute import SimpleImputer

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

imputer = SimpleImputer(strategy='median')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

# Verify no nulls remain
remaining_nulls = df.isnull().sum().sum()
print(f"Remaining nulls after imputation: {remaining_nulls}")
print(f"Final shape: {df.shape}")

Remaining nulls after imputation: 0
Final shape: (307511, 84)


**Save Processed Dataset**

In [8]:
df.to_csv('../data/processed/application_processed.csv', index=False)
print("Saved → data/processed/application_processed.csv")
print(f"Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Quick feature summary
new_features = ['AGE_YEARS', 'EMPLOYMENT_YEARS', 'DAYS_EMPLOYED_ANOMALY',
                'CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 
                'CREDIT_GOODS_RATIO', 'LOAN_TERM_MONTHS',
                'INCOME_PER_PERSON', 'HAS_CAR', 'DOCUMENT_COUNT']

print(f"\nEngineered features added: {len(new_features)}")
for f in new_features:
    print(f"  ✓ {f}")

Saved → data/processed/application_processed.csv
Final dataset: 307,511 rows × 84 columns

Engineered features added: 10
  ✓ AGE_YEARS
  ✓ EMPLOYMENT_YEARS
  ✓ DAYS_EMPLOYED_ANOMALY
  ✓ CREDIT_INCOME_RATIO
  ✓ ANNUITY_INCOME_RATIO
  ✓ CREDIT_GOODS_RATIO
  ✓ LOAN_TERM_MONTHS
  ✓ INCOME_PER_PERSON
  ✓ HAS_CAR
  ✓ DOCUMENT_COUNT
